In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
import numpy as np
import joblib
import os

# -------------------------------
#  Load and clean dataset
# -------------------------------
data = pd.read_csv(r"D:\WildFire Project\data\CA_Weather_Fire_Dataset_1984-2025.csv")

rename_map = {
    "MAX_TEMP": "temperature",
    "PRECIPITATION": "rainfall",
    "AVG_WIND_SPEED": "wind_speed",
    "SEASON": "season",
    "FIRE_START_DAY": "fire_risk"
}
data = data.rename(columns=rename_map)
data = data.dropna(subset=["temperature", "rainfall", "wind_speed", "season", "fire_risk"])
data["fire_risk"] = data["fire_risk"].astype(int)

#  Features and Target
X = data[["temperature", "rainfall", "wind_speed", "season"]]
y = data["fire_risk"]

#  Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
#  Preprocessing
# -------------------------------
numeric_features = ["temperature", "rainfall", "wind_speed"]
categorical_features = ["season"]

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# -------------------------------
#  SMOTE + ENN (resampling + cleaning)
# -------------------------------
smote_enn = SMOTEENN(random_state=42)

# -------------------------------
#  Model
# -------------------------------
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight=None,
    max_depth=12
)

#  Build an Imbalanced-Learn Pipeline
model = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote_enn", smote_enn),
    ("classifier", rf)
])

# -------------------------------
#  Train the model
# -------------------------------
model.fit(X_train, y_train)

# -------------------------------
#  Save model
# -------------------------------
joblib.dump(model, "wildfire_risk_model.pkl")
print("✅ Model trained and saved successfully as wildfire_risk_model.pkl")
print("Model path:", os.path.abspath("wildfire_risk_model.pkl"))

# -------------------------------
#  Model Evaluation
# -------------------------------
y_probs = model.predict_proba(X_test)[:, 1]  # predicted probabilities
default_pred = (y_probs >= 0.5).astype(int)

print("\n--- Default Threshold (0.5) ---")
print("Confusion Matrix:\n", confusion_matrix(y_test, default_pred))
print("\nClassification Report:\n", classification_report(y_test, default_pred))
print("Accuracy:", accuracy_score(y_test, default_pred))
print("ROC AUC:", roc_auc_score(y_test, y_probs))

# -------------------------------
#  Threshold Tuning
# -------------------------------
print("\n--- Threshold Tuning ---")
thresholds = np.arange(0.3, 0.7, 0.05)  # test between 0.3 and 0.65

best_f1 = 0
best_t = 0.5

for t in thresholds:
    y_pred_t = (y_probs >= t).astype(int)
    report = classification_report(y_test, y_pred_t, output_dict=True)
    f1 = report["1"]["f1-score"]
    recall = report["1"]["recall"]
    precision = report["1"]["precision"]
    
    print(f"Threshold: {t:.2f} | Precision: {precision:.2f} | Recall: {recall:.2f} | F1: {f1:.2f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"\n🔥 Best threshold found: {best_t:.2f} with F1-score: {best_f1:.2f}")

# Evaluate again using best threshold
y_best = (y_probs >= best_t).astype(int)
print("\n--- Optimized Threshold Results ---")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_best))
print("\nClassification Report:\n", classification_report(y_test, y_best))
print("Accuracy:", accuracy_score(y_test, y_best))
print("ROC AUC:", roc_auc_score(y_test, y_probs))

✅ Model trained and saved successfully as wildfire_risk_model.pkl
Model path: d:\WildFire Project\notebooks\wildfire_risk_model.pkl

--- Default Threshold (0.5) ---
Confusion Matrix:
 [[1570  432]
 [ 352  642]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.78      0.80      2002
           1       0.60      0.65      0.62       994

    accuracy                           0.74      2996
   macro avg       0.71      0.72      0.71      2996
weighted avg       0.74      0.74      0.74      2996

Accuracy: 0.7383177570093458
ROC AUC: 0.7950859502670369

--- Threshold Tuning ---
Threshold: 0.30 | Precision: 0.57 | Recall: 0.70 | F1: 0.63
Threshold: 0.35 | Precision: 0.57 | Recall: 0.69 | F1: 0.63
Threshold: 0.40 | Precision: 0.58 | Recall: 0.67 | F1: 0.62
Threshold: 0.45 | Precision: 0.58 | Recall: 0.66 | F1: 0.62
Threshold: 0.50 | Precision: 0.60 | Recall: 0.65 | F1: 0.62
Threshold: 0.55 | Precision: 0.61 | Recall: 0.64 | F1: